In [1]:
# 0) ambiente y seeds
#pip install -U "transformers>=4.44" "datasets>=2.19" "accelerate>=0.34" peft bitsandbytes trl
import os, random, math, torch, pandas as pd
SEED = 123
random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

PyTorch: 2.6.0+cu124 | device: cuda


In [2]:
# 1) carga tokenizer + modelo Qwen3-1.7B en 4-bit (sin LoRA aún)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-1.7B"   # base para SFT
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

quant_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16,
)

print("[1] cargando tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)

print("[1] cargando modelo (4-bit)…")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=quant_4bit, trust_remote_code=True
)

MODEL_MAX = getattr(model.config, "max_position_embeddings", 32768)  # Qwen3: 32K
tokenizer.model_max_length = MODEL_MAX
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Para entrenar en 6 GB, usamos longitud menor (2K–4K). Para inferencia podrás usar mucho más.
MAX_SEQ_LEN_TRAIN = 2048
print("[1] MODEL_MAX:", MODEL_MAX, "| MAX_SEQ_LEN_TRAIN:", MAX_SEQ_LEN_TRAIN)


[1] cargando tokenizer…
[1] cargando modelo (4-bit)…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[1] MODEL_MAX: 40960 | MAX_SEQ_LEN_TRAIN: 2048


In [3]:
# 2) preparar k-bit + checkpointing + grads en input (aún sin LoRA)
from peft import prepare_model_for_kbit_training

print("[2] preparando para k-bit training…")
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
print("[2] listo.")

[2] preparando para k-bit training…
[2] listo.


In [4]:
# 3) detectar módulos de atención y aplicar LoRA
from peft import LoraConfig, get_peft_model

# Intento directo para Qwen/LLaMA:
default_targets = ["q_proj","k_proj","v_proj","o_proj"]

# Autodetección por sufijo (por si el checkpoint cambia nombres)
all_module_names = [n for n, _ in model.named_modules()]
selected = [t for t in default_targets if any(n.endswith(t) for n in all_module_names)]
if not selected:
    # fallback: prueba otras variantes comunes
    for group in (["query_key_value","dense"], ["Wqkv","Wo"]):
        found = [t for t in group if any(n.endswith(t) for n in all_module_names)]
        if found:
            selected = found; break
if not selected:
    raise ValueError("[3] No encontré módulos de atención típicos. Ejemplos:", all_module_names[:40])

print("[3] target_modules LoRA:", selected)
lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=selected
)
model = get_peft_model(model, lora_cfg)

# Chequeo: % de parámetros entrenables
def count_trainable_params(m):
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot= sum(p.numel() for p in m.parameters())
    return tr, tot, 100*tr/tot
t, T, pct = count_trainable_params(model)
print(f"[3] trainables: {t:,} / {T:,} ({pct:.4f}%)")


[3] target_modules LoRA: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
[3] trainables: 3,211,264 / 1,019,143,168 (0.3151%)


In [5]:
# 4) datos (CSV) y vistazo rápido
DATA_CSV = "data-sources/pre-processed/data_finetuning.csv"  # columnas: article, summary
df = pd.read_csv(DATA_CSV)
print("[4] columnas:", df.columns.tolist(), "| filas:", len(df))
assert {"article","summary"} <= set(df.columns), "CSV debe tener 'article' y 'summary'"
display(df.head(2))

[4] columnas: ['name', 'article', 'summary'] | filas: 3798


,name,article,summary
0,10.1002-14651858.CD002057.pub4,Background\r\nThis is an update of a review pu...,Inhaled versus systemic corticosteroids for th...
1,10.1002-14651858.CD001746.pub4,Background\r\nChildren's exposure to other peo...,Can interventions for parents and people carin...


In [6]:
# 5) tokenización — elige sin-chunk o con-chunk
USE_CHUNKING = True  # ← pon False si quieres la versión simple truncada a 4K
MAX_TARGET_TOKENS = 384  # ajusta si tu PLS es largo; 384–512 suele ir bien a 4K

SYS_PROMPT = (
  "You are a helpful medical/health writer who rewrites complex biomedical text into Plain Language Summaries "
  "understandable for laypeople. Use short sentences, everyday words, and neutral tone. Avoid jargon; when unavoidable, define it simply."
)

if USE_CHUNKING:
    # ---- CHUNKING (recomendado si artículos son muy largos) ----
    OVERLAP = 128
    PROMPT_PREFIX = SYS_PROMPT + "\n\nScientific Text:\n"
    PROMPT_SUFFIX = "\n\nPlain Language Summary:"

    def tokenize_chunks(article: str, summary: str):
        tgt_ids = tokenizer(summary + tokenizer.eos_token, add_special_tokens=False,
                            truncation=True, max_length=MAX_TARGET_TOKENS).input_ids
        prefix_ids = tokenizer(PROMPT_PREFIX, add_special_tokens=False).input_ids
        suffix_ids = tokenizer(PROMPT_SUFFIX, add_special_tokens=False).input_ids
        avail_for_chunk = MAX_SEQ_LEN_TRAIN - len(tgt_ids) - len(prefix_ids) - len(suffix_ids)
        if avail_for_chunk <= 0:
            ids = tgt_ids[-MAX_SEQ_LEN_TRAIN:]; labels = ids[:]
            return [{"input_ids": ids, "labels": labels}]
        art_ids = tokenizer(article, add_special_tokens=False).input_ids
        stride = max(1, avail_for_chunk - OVERLAP)
        examples = []
        for start in range(0, len(art_ids), stride):
            chunk = art_ids[start:start + avail_for_chunk]
            if not chunk: break
            inp_ids = prefix_ids + chunk + suffix_ids
            ids = inp_ids + tgt_ids
            labels = [-100]*len(inp_ids) + tgt_ids
            examples.append({"input_ids": ids, "labels": labels})
        return examples

    def df_to_dataset(frame: pd.DataFrame):
        from datasets import Dataset
        input_ids_list, labels_list = [], []
        for a, s in zip(frame["article"].tolist(), frame["summary"].tolist()):
            for ex in tokenize_chunks(a, s):
                input_ids_list.append(ex["input_ids"])
                labels_list.append(ex["labels"])
        return Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})

else:
    # ---- SIN CHUNK (simple truncado a 4K) ----
    def build_prompt(article: str) -> str:
        return f"{SYS_PROMPT}\n\nScientific Text:\n{article}\n\nPlain Language Summary:"

    def tokenize_one(article: str, summary: str):
        tgt = tokenizer(summary + tokenizer.eos_token, add_special_tokens=False,
                        truncation=True, max_length=MAX_TARGET_TOKENS).input_ids
        avail = MAX_SEQ_LEN_TRAIN - len(tgt)
        if avail <= 0:
            ids = tgt[-MAX_SEQ_LEN_TRAIN:]; labels = ids[:]
            return {"input_ids": ids, "labels": labels}
        inp = tokenizer(build_prompt(article), add_special_tokens=False,
                        truncation=True, max_length=avail).input_ids
        ids = inp + tgt
        labels = [-100]*len(inp) + tgt
        return {"input_ids": ids, "labels": labels}

    def df_to_dataset(frame: pd.DataFrame):
        from datasets import Dataset
        input_ids_list, labels_list = [], []
        for a, s in zip(frame["article"].tolist(), frame["summary"].tolist()):
            ex = tokenize_one(a, s)
            input_ids_list.append(ex["input_ids"])
            labels_list.append(ex["labels"])
        return Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})

print("[5] tokenizador listo. USE_CHUNKING =", USE_CHUNKING)


[5] tokenizador listo. USE_CHUNKING = True


In [7]:
# 6) split y construcción de datasets tokenizados
from sklearn.model_selection import train_test_split
from datasets import DatasetDict

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, shuffle=True)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, shuffle=True)

print("[6] tokenizando train…")
tok_train = df_to_dataset(train_df)
print("[6] tokenizando val…")
tok_val   = df_to_dataset(val_df)
print("[6] tokenizando test…")
tok_test  = df_to_dataset(test_df)

tok = DatasetDict(train=tok_train, validation=tok_val, test=tok_test)
print(tok)


[6] tokenizando train…
[6] tokenizando val…
[6] tokenizando test…
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3146
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 675
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 671
    })
})


In [8]:
# 7) collator + sanity forward
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

@dataclass
class CausalCollator:
    pad_token_id: int
    def __call__(self, batch):
        ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
        ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
        am = [torch.ones_like(x) for x in ii]
        ii = pad_sequence(ii, batch_first=True, padding_value=self.pad_token_id)
        ll = pad_sequence(ll, batch_first=True, padding_value=-100)
        am = pad_sequence(am, batch_first=True, padding_value=0)
        return {"input_ids": ii, "labels": ll, "attention_mask": am}

collator = CausalCollator(pad_token_id=tokenizer.pad_token_id)
dl = DataLoader(tok["train"], batch_size=1, shuffle=True, collate_fn=collator)

batch = next(iter(dl))
for k,v in batch.items(): print("[7]", k, v.shape, v.dtype)

model.train()
with torch.cuda.amp.autocast(dtype=torch.bfloat16 if bf16_ok else torch.float16):
    out = model(**{k: v.to(model.device) for k,v in batch.items()})
print("[7] loss forward ok:", float(out.loss))



[7] input_ids torch.Size([1, 1071]) torch.int64
[7] labels torch.Size([1, 1071]) torch.int64
[7] attention_mask torch.Size([1, 1071]) torch.int64


C:\Users\jsoa\AppData\Local\Temp\ipykernel_23932\3890302759.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16 if bf16_ok else torch.float16):


[7] loss forward ok: 2.4678423404693604


In [9]:
# 8) sanity backward (un paso manual)
optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
batch = next(iter(dl))
batch = {k: v.to(model.device) for k,v in batch.items()}
optim.zero_grad()
with torch.cuda.amp.autocast(dtype=torch.bfloat16 if bf16_ok else torch.float16):
    out = model(**batch)
loss = out.loss
print("[8] loss:", float(loss), "| requires_grad:", loss.requires_grad)
loss.backward()
optim.step()
print("[8] backward y step OK")

C:\Users\jsoa\AppData\Local\Temp\ipykernel_23932\91581523.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16 if bf16_ok else torch.float16):


[8] loss: 1.5142788887023926 | requires_grad: True
[8] backward y step OK


In [10]:
# 9) Trainer (1 época para validación de pipeline)
from transformers import Trainer, TrainingArguments

EPOCHS = 1
BATCH_SIZE = 1
ACCUM_STEPS = 16
WARMUP_RATIO = 0.1

n_train = len(tok["train"])
steps_per_epoch = max(1, math.ceil(n_train / (BATCH_SIZE * ACCUM_STEPS)))
warmup_steps = int(WARMUP_RATIO * steps_per_epoch * EPOCHS)

args = TrainingArguments(
    output_dir="outputs/qwen3-1p7b-qlora",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=max(1, steps_per_epoch//5),
    do_eval=True,
    eval_steps=steps_per_epoch,
    save_steps=steps_per_epoch,
    save_total_limit=2,
    bf16=bf16_ok,
    fp16=not bf16_ok,
    dataloader_pin_memory=True,
    seed=SEED, data_seed=SEED,
    report_to=None,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    data_collator=collator,
)

print(f"[9] entrenando… pasos/época={steps_per_epoch}")
trainer.train()
print("[9] fin de trainer.train()")


[9] entrenando… pasos/época=197


Step,Training Loss
39,1.806400
78,1.598900
117,1.548600
156,1.584200
195,1.561700


[9] fin de trainer.train()


In [ ]:
#  10) Guardar modelo
# Guarda los pesos finales del modelo ajustado
trainer.save_model("models/qwen3/qwen3-1p7b-qlora-final")

# Guarda el tokenizer para inferencia
tokenizer.save_pretrained("models/qwen3/qwen3-1p7b-qlora-final")

('models/qwen3/qwen3-1p7b-qlora-final\\tokenizer_config.json',
 'models/qwen3/qwen3-1p7b-qlora-final\\special_tokens_map.json',
 'models/qwen3/qwen3-1p7b-qlora-final\\chat_template.jinja',
 'models/qwen3/qwen3-1p7b-qlora-final\\vocab.json',
 'models/qwen3/qwen3-1p7b-qlora-final\\merges.txt',
 'models/qwen3/qwen3-1p7b-qlora-final\\added_tokens.json',
 'models/qwen3/qwen3-1p7b-qlora-final\\tokenizer.json')

In [ ]:
# # 10) Evaluar modelo
trainer.evaluate()

{'eval_loss': 1.566867709159851,
 'eval_runtime': 686.1647,
 'eval_samples_per_second': 0.984,
 'eval_steps_per_second': 0.984,
 'epoch': 1.0}

In [1]:
# 11) Cargar modelo
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# === Configura rutas ===
MODEL_ID = "Qwen/Qwen3-1.7B"
CKPT_DIR = "outputs/qwen3-1p7b-qlora/checkpoint-197"   # tu checkpoint entrenado

# === Carga tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Carga modelo base en 4bit ===
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb_cfg, trust_remote_code=True
)

# === Monta el adaptador LoRA fine-tuneado ===
model = PeftModel.from_pretrained(base_model, CKPT_DIR)
model.eval()

print("✅ Modelo y adaptador cargados correctamente")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Modelo y adaptador cargados correctamente


In [24]:
# 12) prompt de inferencia
SYS_PROMPT = (
    "You are a helpful medical/health writer who rewrites complex biomedical text "
    "into Plain Language Summaries understandable for laypeople. "
    "Use short sentences, everyday words, and a neutral tone."
)

article = """Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim of this review is to examine the recent literature, in terms of physiological mechanisms of EAMC. A comprehensive search was conducted on PubMed and Google Scholar. The following terminology was applied: muscle cramps, neuromuscular hypothesis (or thesis), dehydration hypothesis, Exercise-Associated muscle cramps, nocturnal cramps, muscle spasm, muscle fatigue. From the initial literature of 424 manuscripts, sixty-nine manuscripts were included, analyzed, compared and summarized. Literature analysis indicates that neuromuscular hypothesis may prevails over the initial hypothesis of the dehydration as the trigger event of muscle cramps. New evidence suggests that the action potentials during a muscle cramp are generated in the motoneuron soma, likely accompanied by an imbalance between the rising excitatory drive from the muscle spindles (Ia) and the decreasing inhibitory drive from the Golgi tendon organs. In conclusion, from the latest investigations there seem to be a spinal involvement rather than a peripheral excitation of the motoneurons."""

prompt = f"{SYS_PROMPT}\n\nScientific Text:\n{article}\n\nPlain Language Summary:"

In [25]:

# 13) Tokenizar y generar el resumen (PLS)

# === Tokenización ===
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# === Generación ===
with torch.inference_mode():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=500,           # límite de salida
        temperature=0.7,              # creatividad
        top_p=0.9,                    # nucleus sampling
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

# === Decodificar ===
generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print("\n--- Generación completa ---\n")
print(generated_text)



--- Generación completa ---

You are a helpful medical/health writer who rewrites complex biomedical text into Plain Language Summaries understandable for laypeople. Use short sentences, everyday words, and a neutral tone.

Scientific Text:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim of this review is to examine the recent literature, in terms of physiological mechanisms of EAMC. A comprehensive search was conducted on PubMed and Google Scholar. The following terminology was applied: muscle cramps, neuromuscular hypothesis (or thesis), de

In [26]:
# 14) comparacion inferencia modelo base con respecto al modelo FT
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", trust_remote_code=True)

# === Configuración de generación común ===
gen_kwargs = dict(
    max_new_tokens=384,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

def generate_with(model, prompt: str):
    # detecta el device real del modelo (GPU o CPU)
    try:
        device = model.device
    except AttributeError:
        device = next(model.parameters()).device

    enc = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.inference_mode():
        out = model.generate(**enc, **gen_kwargs)

    # devuelve solo la parte generada (sin el prompt)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [27]:
base_text = generate_with(base, prompt)      # modelo base
ft_text   = generate_with(model, prompt)        # modelo fine-tuneado (LoRA)
print("BASE:\n", base_text)
print("\nFT:\n", ft_text)

BASE:
  
[Your summary here]
Okay, I need to write a plain language summary of the given scientific text about Exercise-Associated Muscle Cramps (EAMC). Let me start by understanding the key points from the original text.

First, EAMC are common, painful muscle spasms related to exercise. The original hypothesis thought it was due to heat cramps, but they retracted that idea. Now, researchers think it's either dehydration/electrolyte issues or a neuromuscular problem. The review looks at recent studies, found 69 papers, and concluded that the neuromuscular theory might be more accurate than dehydration. They mention that during a cramp, motor neurons send signals, and there's an imbalance in excitatory and inhibitory signals from the muscle spindles and Golgi tendon organs. The conclusion is that the issue is in the spinal cord, not just the muscles.

Now, translating this into simple terms. Avoid jargon. Make sure to explain what each part means without technical terms. Use short sent

In [34]:
import pandas as pd

target_text = "Exercise-Associated Muscle Cramps (EAMC) are a common type of muscle spasm, in which a muscle continually contracts without intention, causing pain. Scientists have tried to explain why these cramps happen, but have not been able to. An idea commonly returned to throughout the years is that EAMCs may be caused by heat. Recently, though, more likely explanations are thought to include dehydration, lack of electrolytes, or issues with the nerves connecting to the muscles. The aim of this review is to look at recent research into how and why EAMCs happen. We searched common online resources for papers. For search terms, we used: muscle cramps, neuromuscular hypothesis (or thesis), dehydration hypothesis, Exercise-Associated muscle cramps, nocturnal cramps, muscle spasm, muscle fatigue. The search returned 424 papers. We read and analyzed 69 of them. From the latest evidence, interactions of nerves with muscles explains muscle cramps better than dehydration. Muscle contraction is normally balanced by special cells in the muscles that sense how stretched or contracted the muscles are. Recent findings suggest malfunctions in these sensor cells result in spinal nerves sending unnecessary signals for the muscles to contract. In summary, the signal causing muscles to contract during a spasm seems to come from the spine rather than the nerve endings within the muscles."

df_eval = pd.DataFrame([{
    "article": article,
    "reference": target_text,
    "prediction": ft_text,
}])

df_eval
# Si quieres guardarlo:
# df_eval.to_csv("outputs/eval_single_qwen3.csv", index=False)

,article,reference,prediction
0,Exercise-Associated Muscle Cramps (EAMC) are a...,Exercise-Associated Muscle Cramps (EAMC) are a...,Painful muscle spasms associated with exercise...


In [37]:
# --- 0) deps (si faltan) ---
# !pip install -q evaluate bert-score textstat

import evaluate
import numpy as np
import pandas as pd
import textstat

# =========================================================
# 1) BERTScore: P, R, F1
# =========================================================
preds = [generated_text]
refs  = [target_text]

bertscore = evaluate.load("bertscore")
bs = bertscore.compute(
    predictions=preds,
    references=refs,
    lang="en",                 # <-- usa "es" si tus PLS están en español
    rescale_with_baseline=True # scores en escala más interpretable
)
# bs contiene listas por ejemplo
bs_p, bs_r, bs_f1 = float(bs["precision"][0]), float(bs["recall"][0]), float(bs["f1"][0])
print(f"BERTScore -> P: {bs_p:.4f} | R: {bs_r:.4f} | F1: {bs_f1:.4f}")

# =========================================================
# 2) Legibilidad con tu función (predicciones y referencia)
# =========================================================
def calcular_legibilidad_textstat(preds):
    lang = 'en'  # cambia a 'es' si corresponde
    textstat.set_lang(lang)
    rows = []
    for t in preds:
        t = t or ""
        row = {
            "flesch_reading_ease":  float(textstat.flesch_reading_ease(t)),
            "flesch_kincaid_grade": float(textstat.flesch_kincaid_grade(t)),
        }
        row.update({
            "gunning_fog":              float(textstat.gunning_fog(t)),
            "smog_index":               float(textstat.smog_index(t)) if textstat.sentence_count(t) >= 3 else float("nan"),
            "dale_chall":               float(textstat.dale_chall_readability_score(t)),
            "automated_readability":    float(textstat.automated_readability_index(t)),
            "coleman_liau":             float(textstat.coleman_liau_index(t)),
            "text_standard":            textstat.text_standard(t, float_output=True),
            "num_sentences":            int(textstat.sentence_count(t)),
            "num_words":                int(textstat.lexicon_count(t, removepunct=True)),
            "syllables":                int(textstat.syllable_count(t)),
            "reading_time_sec":         float(textstat.reading_time(t)),
        })
        rows.append(row)

    def _try_mean(key: str):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else float("nan")

    keys = sorted({k for r in rows for k in r.keys()})
    summary = {"n_examples": len(preds), "lang": lang}
    for k in keys:
        summary[f"mean_{k}"] = _try_mean(k)

    per_example = pd.DataFrame(rows)
    return summary, per_example

# Legibilidad de la predicción (modelo) y del target (referencia)
leg_pred_summary, leg_pred_df = calcular_legibilidad_textstat([generated_text])
leg_ref_summary,  leg_ref_df  = calcular_legibilidad_textstat([target_text])

print("\nLegibilidad (PRED):", leg_pred_summary)
print("Legibilidad (REF):",  leg_ref_summary)

# =========================================================
# 3) DataFrame consolidado (1 fila) y opcional export
# =========================================================
df_row = pd.DataFrame([{
    "article": article,
    "reference": target_text,
    "prediction": generated_text,
    "bertscore_precision": bs_p,
    "bertscore_recall":    bs_r,
    "bertscore_f1":        bs_f1,
    # algunos campos de legibilidad útiles
    "pred_flesch_reading_ease":  float(leg_pred_df.loc[0, "flesch_reading_ease"]),
    "pred_flesch_kincaid":       float(leg_pred_df.loc[0, "flesch_kincaid_grade"]),
    "pred_gunning_fog":          float(leg_pred_df.loc[0, "gunning_fog"]),
    "pred_smog_index":           float(leg_pred_df.loc[0, "smog_index"]),
    "pred_dale_chall":           float(leg_pred_df.loc[0, "dale_chall"]),
    "pred_automated_readability":float(leg_pred_df.loc[0, "automated_readability"]),
    "pred_coleman_liau":         float(leg_pred_df.loc[0, "coleman_liau"]),
    "pred_num_sentences":        int(leg_pred_df.loc[0, "num_sentences"]),
    "pred_num_words":            int(leg_pred_df.loc[0, "num_words"]),
    "pred_syllables":            int(leg_pred_df.loc[0, "syllables"]),
    "pred_reading_time_sec":     float(leg_pred_df.loc[0, "reading_time_sec"]),
    # también puedes incluir los del target para comparar:
    "ref_flesch_reading_ease":   float(leg_ref_df.loc[0, "flesch_reading_ease"]),
    "ref_flesch_kincaid":        float(leg_ref_df.loc[0, "flesch_kincaid_grade"]),
}])

print("\nResumen por fila:")
display(df_row.head(1))

# (Opcional) guardar
# df_row.to_csv("outputs/one_eval_row.csv", index=False)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore -> P: 0.0623 | R: 0.3001 | F1: 0.1798

Legibilidad (PRED): {'n_examples': 1, 'lang': 'en', 'mean_automated_readability': 15.786564171122997, 'mean_coleman_liau': 15.752727272727274, 'mean_dale_chall': 11.609126559714795, 'mean_flesch_kincaid_grade': 14.04301247771836, 'mean_flesch_reading_ease': 28.955695187165816, 'mean_gunning_fog': 16.73440285204991, 'mean_num_sentences': 34.0, 'mean_num_words': 660.0, 'mean_reading_time_sec': 56.629949999999994, 'mean_smog_index': 15.444088786983503, 'mean_syllables': 1234.0, 'mean_text_standard': 16.0}
Legibilidad (REF): {'n_examples': 1, 'lang': 'en', 'mean_automated_readability': 12.717714078374456, 'mean_coleman_liau': 13.40943396226415, 'mean_dale_chall': 11.818993613933236, 'mean_flesch_kincaid_grade': 10.139811320754717, 'mean_flesch_reading_ease': 51.41099419448477, 'mean_gunning_fog': 11.617416545718434, 'mean_num_sentences': 13.0, 'mean_num_words': 212.0, 'mean_reading_time_sec': 17.1873, 'mean_smog_index': 11.51311941424646, 'm

,article,reference,prediction,bertscore_precision,bertscore_recall,bertscore_f1,pred_flesch_reading_ease,pred_flesch_kincaid,pred_gunning_fog,pred_smog_index,pred_dale_chall,pred_automated_readability,pred_coleman_liau,pred_num_sentences,pred_num_words,pred_syllables,pred_reading_time_sec,ref_flesch_reading_ease,ref_flesch_kincaid
0,Exercise-Associated Muscle Cramps (EAMC) are a...,Exercise-Associated Muscle Cramps (EAMC) are a...,You are a helpful medical/health writer who re...,0.06232,0.300074,0.179774,28.955695,14.043012,16.734403,15.444089,11.609127,15.786564,15.752727,34,660,1234,56.62995,51.410994,10.139811


In [38]:
!pip install git+https://github.com/google-research/alignscore.git


Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/google-research/alignscore.git to c:\users\jsoa\appdata\local\temp\pip-req-build-17dpcco0


  Running command git clone --filter=blob:none --quiet https://github.com/google-research/alignscore.git 'C:\Users\jsoa\AppData\Local\Temp\pip-req-build-17dpcco0'
  remote: Repository not found.
  fatal: repository 'https://github.com/google-research/alignscore.git/' not found
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/google-research/alignscore.git 'C:\Users\jsoa\AppData\Local\Temp\pip-req-build-17dpcco0' did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/google-research/alignscore.git 'C:\Users\jsoa\AppData\Local\Temp\pip-req-build-17dpcco0' did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
